In [12]:
import sys
import time
sys.path.append('.')  # Asegura que encuentre tu carpeta 'app'

import pandas as pd
import statistics
from app.embeddings.model import get_embedder
from app.vectorstore.qdrant_client import get_qdrant_manager
from app.rag.pipeline import RAGPipeline
from app.core.config import settings

# Forzar a Pandas a mostrar todo el texto de los chunks sin recortarlo feo
pd.set_option('display.max_colwidth', None)

# Instanciar componentes core
embedder = get_embedder()
qdrant_manager = get_qdrant_manager()
rag_pipeline = RAGPipeline()

print("Entorno local, configuraciones de Pandas y modelos listos.")

2026-05-19 11:28:30 | INFO     | app.retrieval.reranker:11 - Reranker inicializado dinámicamente con el modelo: llama3.2:1b
2026-05-19 11:28:30 | INFO     | app.llm.client:21 - Modelo 'llama3.2:1b' verificado — calentando...
2026-05-19 11:28:32 | INFO     | app.llm.client:28 -  Modelo 'llama3.2:1b' caliente en memoria VRAM
2026-05-19 11:28:32 | INFO     | app.rag.pipeline:9 - RAG Pipeline inicializado
Entorno local, configuraciones de Pandas y modelos listos.


In [9]:
query = "¿Qué latencia en el percentil p95 debe mantener el sistema para un buen resultado?"

print(f"PREGUNTA DEL USUARIO:\n'{query}'\n")

start = time.time()
query_vector = embedder.embed_text(query)
print(f"Tiempo de Vectorización: {(time.time() - start)*1000:.2f} ms")
print(f"Dimensiones del Vector: {len(query_vector)}")
print(f"Muestra de los primeros 5 componentes del vector:\n{query_vector[:5]} ...")

PREGUNTA DEL USUARIO:
'¿Qué latencia en el percentil p95 debe mantener el sistema para un buen resultado?'

Tiempo de Vectorización: 4899.15 ms
Dimensiones del Vector: 384
Muestra de los primeros 5 componentes del vector:
[-0.014429960399866104, -0.008432361297309399, -0.03383438661694527, -0.09761295467615128, -0.0818546712398529] ...


In [13]:
TOP_K = 10
NUM_RUNS = 5

print("BUSCANDO EN EL GRAFO HNSW DE QDRANT (MÉTRICA COSENO)...\n")

#WARMUP DEL INDICE
qdrant_manager.client.search(
    collection_name=settings.QDRANT_COLLECTION_NAME,
    query_vector=query_vector,
    limit=TOP_K,
    with_payload=False
)

print("Warmup completado.\n")

#Benchmark puro del indice vectorial
latencias = []

for run in range(NUM_RUNS):

    start_hnsw = perf_counter()

    results_pure = qdrant_manager.client.search(
        collection_name=settings.QDRANT_COLLECTION_NAME,
        query_vector=query_vector,
        limit=TOP_K,
        with_payload=False
    )

    latency_ms = (perf_counter() - start_hnsw) * 1000

    latencias.append({
        "run": run + 1,
        "latency_ms": round(latency_ms, 2)
    })

# DataFrame benchmark
df_benchmark = pd.DataFrame(latencias)

# Estadísticas
avg_latency = df_benchmark["latency_ms"].mean()
min_latency = df_benchmark["latency_ms"].min()
max_latency = df_benchmark["latency_ms"].max()

summary_df = pd.DataFrame([{
    "top_k": TOP_K,
    "runs": NUM_RUNS,
    "avg_latency_ms": round(avg_latency, 2),
    "min_latency_ms": round(min_latency, 2),
    "max_latency_ms": round(max_latency, 2),
    "status": "CUMPLE" if avg_latency < 100 else "NO CUMPLE"
}])

print("BENCHMARK HNSW")
display(df_benchmark)

print("RESUMEN ESTADÍSTICO")
display(summary_df)

#RECUPERACIÓN COMPLETA PARA EL FLUJO RAG
results_with_text = qdrant_manager.search(
    query_vector=query_vector,
    top_k=TOP_K
)

# 4. VISUALIZACIÓN DE RESULTADOS RAG
rows = []

for idx, hit in enumerate(results_with_text, 1):

    clean_text = (
        hit["text"]
        .replace("\n", " ")
        .replace("\xa0", " ")
        .strip()
    )

    rows.append({
        "rank": idx,
        "score": round(hit["score"], 4),
        "similarity_percent": round(hit["score"] * 100, 2),
        "filename": hit["filename"],
        "chunk_index": hit["chunk_index"],
        "chars": len(clean_text),
        "words": len(clean_text.split()),
        "text_preview": clean_text[:300]
    })

df_results = pd.DataFrame(rows)

print("FRAGMENTOS RECUPERADOS")
display(df_results)


# 5. CONTEXTO FINAL INYECTADO AL LLM
contexto_inyectado = "\n\n".join(
    [row["text_preview"] for row in rows]
)

prompt_final = (
    "SISTEMA: Eres un asistente de IA estricto.\n\n"
    f"CONTEXTO:\n{contexto_inyectado}\n\n"
    f"PREGUNTA: {query}"
)

prompt_df = pd.DataFrame([{
    "query": query,
    "context_length_chars": len(contexto_inyectado),
    "prompt_preview": prompt_final[:500]
}])

print("PROMPT FINAL")
display(prompt_df)

BUSCANDO EN EL GRAFO HNSW DE QDRANT (MÉTRICA COSENO)...

Warmup completado.

BENCHMARK HNSW


,run,latency_ms
0,1,13.83
1,2,10.26
2,3,11.46
3,4,8.50
4,5,8.65


RESUMEN ESTADÍSTICO


,top_k,runs,avg_latency_ms,min_latency_ms,max_latency_ms,status
0,10,5,10.54,8.5,13.83,CUMPLE


FRAGMENTOS RECUPERADOS


,rank,score,similarity_percent,filename,chunk_index,chars,words,text_preview
0,1,0.6095,60.95,rubrica.entregable.semana.02.pdf,9,495,62,"AG típicas (utilizando vectores estándar de 1536 dimensiones y buscando las 10 coincidencias principales o top-K 10), el sistema debe lograr una tasa de recuerdo (recall) del 90 al 95, manteniendo una latencia en el percentil 95"
1,2,0.5576,55.76,rubrica.entregable.semana.02.pdf,4,493,58,ectorial El componente más crítico del sistema en esta fase es la base de datos vectorial. La decisión sobre qué motor implementar define la complejidad del despliegue y la latencia del sistema. Análisis de Opciones Arquitectónica
2,3,0.5413,54.13,rubrica.entregable.semana.02.pdf,13,678,76,"ñadiendo información externa, el puntaje debe penalizarse. 4. Relevancia de la Respuesta (Answer Relevancy): Califica qué tan directamente la respuesta del LLM aborda la inquietud original del usuario, penalizando las divagaciones. Dinámi"
3,4,0.5139,51.39,rubrica.entregable.semana.02.pdf,20,644,72,ón arquitectónica justificada. Cumple la meta de buen resultado: recall 90 y latencias submilisegundo ( 100ms). Base de datos persistente ejecutando búsquedas puramente vectoriales. Recuperación precisa en la gran mayoría de las Conf
4,5,0.4935,49.35,rubrica.entregable.semana.02.pdf,23,495,57,"ticamente el límite de tokens de los modelos de embedding o del LLM. Métricas RAG y Contención de Alucinaciones Implementación de métricas de evaluación (Fidelidad, Relevancia, Precisión) con resultados excelentes. El System Prompt re"
5,6,0.4922,49.22,rubrica.entregable.semana.02.pdf,15,495,61,"éxito es la correcta ingesta y la persistencia de los índices. 2. Ingeniero de Modelos LLM y Lógica Cognitiva: Configura los servicios locales (Ollama, LMStudio), selecciona los modelos base y redacta los System Prompts restrictivos"
6,7,0.4756,47.56,rubrica.entregable.semana.02.pdf,8,497,58,"tos. Inyectar textos crudos comprometerá la respuesta del LLM. Qué constituye un buen resultado? Un resultado exitoso en la implementación técnica de la base de datos vectorial no es solo que el sistema ""devuelva respuestas"". Un"
7,8,0.4474,44.74,rubrica.entregable.semana.02.pdf,10,493,57,"el LLM son altamente relevantes a la consulta del usuario y 100 fieles al texto de la base de datos, sin generar alucinaciones. Evaluación del Sistema RAG (El Conocimiento Estático) La evaluación de la recuperación de informaci"
8,9,0.4452,44.52,rubrica.entregable.semana.02.pdf,2,494,58,de datos elegida. En esta fase es vital configurar la indexación: los equipos deben seleccionar una métrica de distancia (como la similitud del coseno) que sea compatible con su modelo de incrustación y ajustar los parámetros de
9,10,0.4039,40.39,rubrica.entregable.semana.02.pdf,19,498,58,neral para evaluar la adopción de la base de datos vectorial y el flujo RAG de manera independiente a la temática del proyecto. Categoría Evaluada 4 - Sobresaliente (Arquitectura Robusta) 3 - Competente (Prototipo Sólido) 2 -


PROMPT FINAL


,query,context_length_chars,prompt_preview
0,¿Qué latencia en el percentil p95 debe mantener el sistema para un buen resultado?,3018,"SISTEMA: Eres un asistente de IA estricto.\n\nCONTEXTO:\nAG típicas (utilizando vectores estándar de 1536 dimensiones y buscando las 10 coincidencias principales o top-K 10), el sistema debe lograr una tasa de recuerdo (recall) del 90 al 95, manteniendo una latencia en el percentil 95 \n\nectorial El componente más crítico del sistema en esta fase es la base de datos vectorial. La decisión sobre"


In [14]:
# Reconstruir la lista de contextos asegurando el formato lineal horizontal
contextos_limpios = []
for hit in results_with_text:
    texto_lineal = " ".join(hit['text'].split()).strip()
    contextos_limpios.append(f"• [Ref: {hit['filename']} (Chunk {hit['chunk_index']})]: {texto_lineal}")

contexto_inyectado = "\n\n".join(contextos_limpios)

system_prompt = (
    "SISTEMA: Eres un asistente de IA estricto. Responde la pregunta del usuario utilizando UNICAMENTE "
    "el contexto proporcionado abajo. Si la respuesta no viene explicitamente en el contexto, "
    "di de forma honesta 'No encontre informacion suficiente en el documento'. NO inventes nada."
)

prompt_final = (
    f"{system_prompt}\n\n"
    f"CONTEXTO RECUPERADO:\n{contexto_inyectado}\n\n"
    f"PREGUNTA DEL USUARIO: {query}\n\n"
    "RESPUESTA DE LA IA:"
)

# Mostrar la tabla estructural del Prompt para el video de la entrega
df_prompt_view = pd.DataFrame({
    "Bloque Operativo": ["1. System Role", "2. Vector Context (Inyectado)", "3. User Query"],
    "Estado de Inferencia": ["Estricto - Anti Alucinaciones", f"Inyectados {len(results_with_text)} fragmentos reformateados.", query]
})

print("CONTROL DE COMPONENTES DEL PROMPT:")
display(df_prompt_view)

print("\nVISTA PREVIA DEL PROMPT COMPLETO CORREGIDO EN HORIZONTAL:")
print("=" * 75)
print(prompt_final[:850] + "\n\n... [TEXTO REESTRUCTURADO CORRECTAMENTE PARA EL LLM] ...")
print("=" * 75)

CONTROL DE COMPONENTES DEL PROMPT:


,Bloque Operativo,Estado de Inferencia
0,1. System Role,Estricto - Anti Alucinaciones
1,2. Vector Context (Inyectado),Inyectados 10 fragmentos reformateados.
2,3. User Query,¿Qué latencia en el percentil p95 debe mantener el sistema para un buen resultado?



VISTA PREVIA DEL PROMPT COMPLETO CORREGIDO EN HORIZONTAL:
SISTEMA: Eres un asistente de IA estricto. Responde la pregunta del usuario utilizando UNICAMENTE el contexto proporcionado abajo. Si la respuesta no viene explicitamente en el contexto, di de forma honesta 'No encontre informacion suficiente en el documento'. NO inventes nada.

CONTEXTO RECUPERADO:
• [Ref: rubrica.entregable.semana.02.pdf (Chunk 9)]: AG típicas (utilizando vectores estándar de 1536 dimensiones y buscando las 10 coincidencias principales o top-K 10), el sistema debe lograr una tasa de recuerdo (recall) del 90 al 95, manteniendo una latencia en el percentil 95 (p95) por debajo de los 100 milisegundos. 1 A nivel cognitivo, un buen resultado significa que las respuestas del LLM son altamente relevantes a la

• [Ref: rubrica.entregable.semana.02.pdf (Chunk 4)]: ectorial El componente más crítico del sistema en esta fase es la

... [TEXTO REESTRUCTURADO CORRECTAMENTE PARA EL LLM] ...


In [15]:
print("EJECUTANDO INFERENCIA LOCAL EN OLLAMA (llama3.2:1b)...")
start_llm = time.time()

rag_response = rag_pipeline.query(query)

print(f"Tiempo total de respuesta cognitiva: {time.time() - start_llm:.2f} segundos\n")
print("RESPUESTA FINAL DE LA RÚBRICA:")
print("-" * 75)
print(rag_response["answer"])
print("-" * 75)

EJECUTANDO INFERENCIA LOCAL EN OLLAMA (llama3.2:1b)...
2026-05-19 11:31:35 | INFO     | app.rag.chain:19 - === RAG Chain iniciada | query: '¿Qué latencia en el percentil p95 debe mantener el sistema para un buen resultado?' ===
2026-05-19 11:31:35 | INFO     | app.retrieval.search:17 - Buscando: '¿Qué latencia en el percentil p95 debe mantener el sistema para un buen resultado?' | top_k=10
2026-05-19 11:31:39 | INFO     | app.retrieval.search:25 - Encontrados 10 fragmentos relevantes
2026-05-19 11:31:39 | DEBUG    | app.retrieval.search:27 -   [1] score=0.6095 | rubrica.entregable.semana.02.pdf | chunk 9
2026-05-19 11:31:39 | DEBUG    | app.retrieval.search:27 -   [2] score=0.5576 | rubrica.entregable.semana.02.pdf | chunk 4
2026-05-19 11:31:39 | DEBUG    | app.retrieval.search:27 -   [3] score=0.5413 | rubrica.entregable.semana.02.pdf | chunk 13
2026-05-19 11:31:39 | DEBUG    | app.retrieval.search:27 -   [4] score=0.5139 | rubrica.entregable.semana.02.pdf | chunk 20
2026-05-19 11:31: